**Install required Libraries**

!pip install -U langchain langchain-classic langchain-community langchain-text_splitters sentence-transformers faiss-cpu transformers requests

**Import required Libraries**

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from transformers import pipeline
import requests


**Sample Document**

In [18]:
docs = [

    "LangChain is a framework for developing LLM applications.",

    "ProjectPro is great for data science project solutions.",

    "FAISS is a library for efficient similarity search."

]

**Split the Text**

In [19]:
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=10)
split_docs = text_splitter.create_documents(docs)

**Embed the Text**

In [20]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(split_docs,embedding)
retriever = db.as_retriever()

C:\Users\prate\AppData\Local\Temp\ipykernel_13316\1645675613.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\prate\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prate\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Retrieve the Data for context**

In [22]:
# ======Step 3: Retrieve Data for context ===

query = "What is ProjectPro?"
retrieved_docs = retriever.invoke(query)
retrieved_text = "\n".join([doc.page_content for doc in retrieved_docs])

**Send the query to MCP and Get the MCP response**

In [25]:
mcp_response = requests.post("http://127.0.0.1:8000/mcp/query", json={"query" : query})
mcp_text = mcp_response.json().get("response","")

**Combine RAG Context + MCP Response**

In [26]:
final_context = f"Retrieved:\n{retrieved_text}\n\nMCP Mmeory:\n{mcp_text}"

**Generate Final Answer Using Hugging Face Model**

In [30]:
hf_pipeline = pipeline("text-generation", model="google/flan-t5-base",max_new_tokens=100)
llm = HuggingFacePipeline(pipeline=hf_pipeline)
#response = llm.predict(f"Answer this query: {query}\n\n{final_context}")
response = llm.invoke(
    f"Answer the question using the context.\n\n"
    f"Question: {query}\n\n"
    f"Context:\n{final_context}"
)
print(response)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Answer the question using the context.

Question: What is ProjectPro?

Context:
Retrieved:
ProjectPro is great for data science project solutions.
LangChain is a framework for developing LLM applications.
FAISS is a library for efficient similarity search.

MCP Mmeory:
ProjectPro is a platform offering solved end-to-end data science and AI projects.
